# 第17集：Pandas合并merge

> 原视频 P17：3.7 pandas 合并 merge｜时长：18分19秒

本笔记严格依照原聊天记录中的讲解顺序整理。每个知识点先解释含义，再给出代码和记录中的预期输出；标为旧写法或故意报错的片段只用于阅读，不作为可执行单元。

## 运行前准备

原聊天记录在这里沿用前一集已经导入的库。为了让本 Notebook 能独立运行，先补上相同的导入。


In [ ]:
import numpy as np
import pandas as pd


原视频内容：[Pandas merge](https://mofanpy.com/tutorials/data-manipulation/np-pd/pd-merge)

`merge` 可以理解成数据库中的“根据共同字段连接表格”。

区别：

```text
concat → 直接上下或左右拼接
merge  → 根据共同键值寻找对应关系
```

## 1. 根据一个key合并


In [ ]:
left = pd.DataFrame({
    "key": ["K0", "K1", "K2", "K3"],
    "A": ["A0", "A1", "A2", "A3"],
    "B": ["B0", "B1", "B2", "B3"]
})

right = pd.DataFrame({
    "key": ["K0", "K1", "K2", "K3"],
    "C": ["C0", "C1", "C2", "C3"],
    "D": ["D0", "D1", "D2", "D3"]
})

result = pd.merge(
    left,
    right,
    on="key"
)

print(result)


输出：

```text
  key   A   B   C   D
0  K0  A0  B0  C0  D0
1  K1  A1  B1  C1  D1
2  K2  A2  B2  C2  D2
3  K3  A3  B3  C3  D3
```

`on="key"` 表示：

> 用左右两张表的key列匹配。

例如：

```text
左表key=K1的行
        +
右表key=K1的行
        ↓
合成一行
```

---

## 2. 更现实的例子

学生表：


In [ ]:
students = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["小明", "小红", "小刚"]
})


成绩表：


In [ ]:
scores = pd.DataFrame({
    "id": [2, 3, 4],
    "score": [90, 85, 100]
})


两张表：

```text
students             scores

id  name             id  score
1   小明              2    90
2   小红              3    85
3   小刚              4   100
```

共同字段是：

```text
id
```

---

## 3. `inner`：只保留左右都有的数据


In [ ]:
result = pd.merge(
    students,
    scores,
    on="id",
    how="inner"
)

print(result)


输出：

```text
   id name  score
0   2   小红     90
1   3   小刚     85
```

只有 `id=2` 和 `id=3` 在两张表中都存在。

```text
inner → 交集
```

`how` 不写时，默认就是：

```python
how="inner"
```

---

## 4. `left`：以左表为主


In [ ]:
result = pd.merge(
    students,
    scores,
    on="id",
    how="left"
)

print(result)


输出：

```text
   id name  score
0   1   小明    NaN
1   2   小红   90.0
2   3   小刚   85.0
```

左表中的所有学生都保留。

小明的 `id=1` 在成绩表中找不到，所以成绩是：

```text
NaN
```

---

## 5. `right`：以右表为主


In [ ]:
result = pd.merge(
    students,
    scores,
    on="id",
    how="right"
)

print(result)


输出：

```text
   id name  score
0   2   小红     90
1   3   小刚     85
2   4  NaN    100
```

右表中的所有成绩记录都保留。

`id=4` 找不到姓名，所以姓名是 `NaN`。

---

## 6. `outer`：左右数据全部保留


In [ ]:
result = pd.merge(
    students,
    scores,
    on="id",
    how="outer"
)

print(result)


输出：

```text
   id name  score
0   1   小明    NaN
1   2   小红   90.0
2   3   小刚   85.0
3   4  NaN   100.0
```

总结：

| how | 保留什么 |
|---|---|
| `inner` | 左右都有的键 |
| `left` | 左边所有键 |
| `right` | 右边所有键 |
| `outer` | 左右全部键 |

---

## 7. 根据多个key合并


In [ ]:
left = pd.DataFrame({
    "班级": ["一班", "一班", "二班"],
    "姓名": ["小明", "小红", "小刚"],
    "语文": [90, 85, 88]
})

right = pd.DataFrame({
    "班级": ["一班", "二班", "二班"],
    "姓名": ["小明", "小刚", "小红"],
    "数学": [80, 92, 95]
})

result = pd.merge(
    left,
    right,
    on=["班级", "姓名"],
    how="inner"
)

print(result)


输出：

```text
   班级  姓名  语文  数学
0  一班  小明  90  80
1  二班  小刚  88  92
```

只有“班级和姓名都相同”才算匹配。

---

## 8. `indicator=True`查看数据来源


In [ ]:
result = pd.merge(
    students,
    scores,
    on="id",
    how="outer",
    indicator=True
)

print(result)


输出：

```text
   id name  score      _merge
0   1   小明    NaN   left_only
1   2   小红   90.0        both
2   3   小刚   85.0        both
3   4  NaN   100.0  right_only
```

`_merge`的含义：

| 值 | 含义 |
|---|---|
| `left_only` | 只来自左表 |
| `right_only` | 只来自右表 |
| `both` | 左右表都有 |

这在检查数据为什么没有匹配上时非常有用。

---

## 9. 根据索引合并


In [ ]:
left = pd.DataFrame(
    {
        "A": ["A0", "A1", "A2"],
        "B": ["B0", "B1", "B2"]
    },
    index=["K0", "K1", "K2"]
)

right = pd.DataFrame(
    {
        "C": ["C0", "C2", "C3"],
        "D": ["D0", "D2", "D3"]
    },
    index=["K0", "K2", "K3"]
)

result = pd.merge(
    left,
    right,
    left_index=True,
    right_index=True,
    how="outer"
)

print(result)


输出：

```text
      A    B    C    D
K0   A0   B0   C0   D0
K1   A1   B1  NaN  NaN
K2   A2   B2   C2   D2
K3  NaN  NaN   C3   D3
```

参数：

```text
left_index=True  → 左表使用索引作为key
right_index=True → 右表使用索引作为key
```

---

## 10. 重名列和suffixes


In [ ]:
boys = pd.DataFrame({
    "key": ["K0", "K1"],
    "age": [18, 19]
})

girls = pd.DataFrame({
    "key": ["K0", "K1"],
    "age": [17, 18]
})

result = pd.merge(
    boys,
    girls,
    on="key",
    suffixes=("_boy", "_girl")
)

print(result)


输出：

```text
  key  age_boy  age_girl
0  K0       18        17
1  K1       19        18
```

因为左右表都有 `age`，所以通过：

```python
suffixes=("_boy", "_girl")
```

加后缀区分。

Pandas官方的 `merge` 接口支持 `how`、`on`、索引连接、`suffixes`、`indicator` 等参数。[Pandas merge文档](https://pandas.pydata.org/docs/reference/api/pandas.merge.html)

---

## 11. 重复key会发生什么


In [ ]:
left = pd.DataFrame({
    "key": ["K0", "K0"],
    "left_value": [1, 2]
})

right = pd.DataFrame({
    "key": ["K0", "K0"],
    "right_value": [10, 20]
})

print(pd.merge(left, right, on="key"))


输出：

```text
  key  left_value  right_value
0  K0           1           10
1  K0           1           20
2  K0           2           10
3  K0           2           20
```

左边两个K0，右边两个K0，会进行全部组合：

```text
2 × 2 = 4行
```

这是多对多连接，可能导致数据行数突然增加。

---
